### Partition Tutorial
[Video](https://youtu.be/761SQ9Hxbic?list=PLeo1K3hjS3uu7dS3Cx0X796sWUjjBFCuV&t=4801)

In [0]:
df = spark.table("workspace.default.movies")
display(df.limit(5))

In [0]:
%sql
select distinct studio from workspace.default.movies

### Partition by Key

In [0]:
# Partition by a key (better data grouping for future groupBy/join on that key)
rep_by_key = df.repartition(6, "studio")  # Exchange hashpartitioning(studio, 6)
rep_by_key.explain("formatted")

In [0]:
display(df)

### Partition in Round Robin Fashion

In [0]:
# 1) Make 6 compute partitions (round-robin shuffle)
rep_rr = df.repartition(6)
rep_rr.count()

In [0]:
rep_rr.explain("formatted")

In [0]:
out_path = "/Volumes/workspace/default/repartition_demo/repartition_6"
rep_rr.write.mode("overwrite").parquet(out_path)

Repartition by key helps if we have multiple per-studio operations such as below

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

# One-time shuffle:
# if we don't do the repartition first, we just use 'df' first, it will be slow 
base = df.repartition(6, "studio")

# Reuse partitioning on the same detailed rows:
agg    = base.groupBy("studio").agg(F.avg(F.col("revenue").cast("double")))
ranked = base.withColumn("rnk", F.row_number().over(Window.partitionBy("studio").orderBy(F.desc("revenue"))))

## Partition and Parallelism: Coalesce
[video](https://youtu.be/761SQ9Hxbic?list=PLeo1K3hjS3uu7dS3Cx0X796sWUjjBFCuV&t=5647)